In [3]:
import pandas as pd
import sqlite3

In [4]:
conn = sqlite3.connect("dataset/rideshare.db")

In [10]:
df = pd.read_sql_query("SELECT status FROM trips LIMIT 5", conn)

In [11]:
df

,status
0,completed
1,cancelled
2,completed
3,completed
4,completed


In [12]:
conn.close()

In [41]:
import google.generativeai as genai
import os
from dotenv import load_dotenv

In [42]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

In [29]:
print("Models available:")
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

Models available:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-

In [18]:
def get_database_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    conn.close()
    schema = "\n\n".join([table[0] for table in tables if table[0] is not None])
    return schema

In [19]:
db_schema = get_database_schema("dataset/rideshare.db")

In [34]:
def ask_database(user_question):

    system_prompt = f"""
    You are an expert SQL Assistant. Your job is to convert natural language questions into SQL queries based on the following SQLite database schema:
    SCHEMA:
    {db_schema}
    RULES:
    1. If the question can be answered using the tables in the schema, return ONLY the raw SQL query. Do not include markdown formatting (like ```sql), do not include explanations. Just the query.
    2. If the question is general English, unrelated to the schema, or impossible to answer with the provided tables, you must reply EXACTLY with:
       "I'm sorry. Your question is invalid or unrelated to the database. I don't know the answer."
    """

    model = genai.GenerativeModel(
        model_name="gemini-3-flash-preview",
        system_instruction=system_prompt
    )
    
    response = model.generate_content(user_question)
    result = response.text.strip()
    
    return result

In [37]:
# Valid Question
valid_q = "How many total users are there in the database?"
print("Question:", valid_q)
print("LLM Output:", ask_database(valid_q))

Question: How many total users are there in the database?
LLM Output: SELECT COUNT(*) FROM users;


In [38]:
# Complex Valid Question"
complex_q = "List the names of the top 3 drivers based on their rating."
print("Question:", complex_q)
print("LLM Output:", ask_database(complex_q))

Question: List the names of the top 3 drivers based on their rating.
LLM Output: SELECT users.name FROM users JOIN drivers ON users.user_id = drivers.user_id ORDER BY drivers.rating DESC LIMIT 3;


In [39]:
# Invalid Question
invalid_q = "What is the capital of France?"
print("Question:", invalid_q)
print("LLM Output:", ask_database(invalid_q))

Question: What is the capital of France?
LLM Output: I'm sorry. Your question is invalid or unrelated to the database. I don't know the answer.
